In [ ]:
# SETUP - safe to re-run. Re-clones the CODE only; never touches result files.
%cd /kaggle/working
!rm -rf StatTrap_SLM
!git clone https://github.com/mirzayasirabdullahbaig07/StatTrap_SLM.git
%cd /kaggle/working/StatTrap_SLM
!git log -1 --oneline
!grep -q "direct_mode" src/stattrap/cli.py && echo "CODE OK: latest version" || echo "!!! OLD CODE ON GITHUB - push your latest commit from your PC, then re-run this cell"
!echo "commit above must be the one you just pushed" 
!pip install -q -e . accelerate bitsandbytes

In [ ]:
# SMOKE TEST (2-3 min, 0.5B). Check parsing before starting the long notebooks.
%cd /kaggle/working/StatTrap_SLM
!rm -f /kaggle/working/smoke.csv /kaggle/working/smoke.manifest.json
!stattrap run --backend hf --models Qwen/Qwen2.5-0.5B-Instruct --per-family 5 --perms 2 --limit 33 --max-new-tokens 1024 --direct-mode logit --batch-size 16 --out /kaggle/working/smoke.csv
import pandas as pd
d = pd.read_csv('/kaggle/working/smoke.csv', keep_default_na=False)
print(d.groupby('style').agg(n=('correct','size'), unparsed=('chosen', lambda s: (s=='').mean()), trunc=('truncated','mean'), acc=('correct','mean'), sec_per_row=('latency_s','mean')))
for r in d.response.head(4): print('---', r[:300])
print('\nNOTE: smoke items are mixed across families, so every batch contains a slow item; sec_per_row here OVERSTATES real speed.')
print('Unparsed DIRECT answers (are these genuine format failures?):')
for r in d[(d['style']=='direct') & (d.chosen=='')].response.head(5): print('  >', r[:200])
